![Health-Informatics: A Python Tutorial](../../Image/chapter-banner.png)


# Module 16: Emerging Directions



**Health Informatics in Python** · Part IV: Advanced Topics · Module 16 of 16

---



The final module looks at where health informatics is heading: **large language
models** for documentation and coding, **FHIR + generative AI** patterns,
**agentic AI** (plan-over-tools over a claims book), **federated** approaches
that analyze data without moving it, and the **ethics, regulation, and equity**
questions that decide whether any of it should ship.


## Learning objectives

By the end of this module you will be able to:

1. Describe how **LLMs** are applied to clinical documentation and **autocoding**.
2. Implement the **extract → structure → validate** pattern for LLM-assisted coding
   (with a deterministic mock so it runs offline).
3. Explain the **FHIR + generative AI** integration pattern.
4. Build a small **agentic** controller: named tools, a fixed plan, intent routing,
   and an audit trail — with the LLM allowed only to *rephrase* numbers.
5. Demonstrate a **federated** computation that shares statistics, not records.
6. Run a basic **equity/bias** check on subgroup performance.


## Dataset

The synthetic EHR plus the clinical-note style text from Module 10. The "LLM" here
is a **deterministic mock** so the notebook runs without external API calls; the
*pattern* is identical to a real LLM integration.

The **agentic** section uses a compact synthetic **claims book** (the shape of
Module 13's member-year table) so the controller has tools to call — still
offline, still no PHI.


In [1]:
# --- Self-contained synthetic EHR generator (identical to earlier parts) ---
import numpy as np
import pandas as pd

# This function generates a synthetic electronic health record (EHR) dataset with several tables.
def make_synthetic_ehr(n_patients=200, seed=42):
    rng = np.random.default_rng(seed)  # Random number generator for reproducibility
    
    # Define some sample names
    first = ["Ava","Liam","Noah","Mia","Zoe","Omar","Ivan","Sara","Leo","Nina",
             "Ruth","Kai","Yara","Theo","Ida","Sam","Ana","Eli","Rex","Uma"]
    last  = ["Khan","Ortiz","Chen","Diaz","Patel","Ali","Brown","Nash","Reed","Vega",
             "Cole","Frost","Grant","Hale","Iqbal","Jain","Kerr","Lund","Mora","Park"]
    
    # Generate patient demographics
    ages  = rng.integers(18, 90, size=n_patients)
    patients = pd.DataFrame({
        "patient_id": [f"P{1000+i}" for i in range(n_patients)],
        "given_name": rng.choice(first, size=n_patients),
        "family_name": rng.choice(last, size=n_patients),
        "sex": rng.choice(["male","female"], size=n_patients, p=[0.49,0.51]),
        "age": ages,
        "birth_year": 2026 - ages,
    })

    # Generate random healthcare encounters for each patient
    enc_rows, enc_types = [], ["ambulatory","emergency","inpatient","wellness"]
    for pid in patients["patient_id"]:
        for _ in range(rng.integers(1, 5)):  # Each patient gets 1-4 encounters
            day = rng.integers(0, 365*3)  # Random day in a 3-year window
            enc_rows.append({
                "encounter_id": f"E{len(enc_rows)+1:05d}",
                "patient_id": pid,
                "encounter_type": rng.choice(enc_types, p=[0.55,0.15,0.10,0.20]),
                "date": (pd.Timestamp("2023-01-01") + pd.Timedelta(days=int(day))).date()
            })
    encounters = pd.DataFrame(enc_rows)

    # Generate observations for each encounter
    obs_defs = [
        ("Body height","cm",150,195),
        ("Body weight","kg",50,110),
        ("Systolic blood pressure","mmHg",100,165),
        ("Heart rate","/min",55,100),
        ("Hemoglobin A1c","%",4.8,9.5)
    ]
    obs_rows = []
    for _, e in encounters.iterrows():
        for name, unit, lo, hi in obs_defs:
            if rng.random() < 0.7:  # 70% chance that the observation appears for the encounter
                obs_rows.append({
                    "observation_id": f"O{len(obs_rows)+1:06d}",
                    "encounter_id": e["encounter_id"],
                    "patient_id": e["patient_id"],
                    "observation": name,
                    "value": round(float(rng.uniform(lo,hi)),1),
                    "unit": unit,
                    "date": e["date"]
                })
    observations = pd.DataFrame(obs_rows)
    
    # Generate conditions (diagnoses) for each patient
    cond_pool = [
        "Essential hypertension","Type 2 diabetes mellitus","Asthma",
        "Acute bronchitis","Major depressive disorder","Osteoarthritis",
        "Chronic kidney disease","Anemia"
    ]
    cond_rows = []
    for pid in patients["patient_id"]:
        # Each patient has 0–3 random conditions
        for c in rng.choice(cond_pool, size=rng.integers(0,4), replace=False):
            cond_rows.append({
                "condition_id": f"C{len(cond_rows)+1:05d}",
                "patient_id": pid,
                "condition": c
            })
    conditions = pd.DataFrame(cond_rows)

    # Generate medications for each patient
    med_pool = [
        "Lisinopril","Metformin","Albuterol","Atorvastatin","Sertraline",
        "Amoxicillin","Ibuprofen","Hydrochlorothiazide"
    ]
    med_rows = []
    for pid in patients["patient_id"]:
        # Each patient takes 0–3 random medications
        for m in rng.choice(med_pool, size=rng.integers(0,4), replace=False):
            med_rows.append({
                "medication_id": f"M{len(med_rows)+1:05d}",
                "patient_id": pid,
                "medication": m
            })
    medications = pd.DataFrame(med_rows)
    
    # Return all generated tables in a dictionary
    return {
        "patients": patients,
        "encounters": encounters,
        "observations": observations,
        "conditions": conditions,
        "medications": medications
    }

# Generate the synthetic dataset and print the number of rows in each table.
ehr = make_synthetic_ehr()
print("Tables:", ", ".join(f"{k} ({len(v)})" for k,v in ehr.items()))


Tables: patients (200), encounters (511), observations (1804), conditions (304), medications (276)


## 16.1 LLMs for clinical documentation and coding

Large Language Models (LLMs) are increasingly utilized in clinical documentation workflows
for a range of applications:

- **Ambient scribing:** Transforming real-time physician-patient conversations into structured clinical notes.
- **Summarization:** Creating concise summaries from lengthy, complex patient charts to assist clinician review.
- **Autocoding:** Automatically suggesting diagnostic (ICD) or clinical finding (SNOMED) codes from unstructured text, e.g., physician notes.

A robust and safe engineering approach for integrating LLM outputs in healthcare follows this pipeline:

1. **Extract** — The LLM identifies relevant information or codes from the source text.
2. **Structure** — The extracted outputs are converted into a predefined, structured format (such as JSON with explicit fields).
3. **Validate** — Deterministic rules or lookups check whether the suggested outputs (e.g., codes) are syntactically valid, allowed, and sufficiently confident.
4. **Human review** — Qualified clinicians review, accept, or edit the LLM’s proposals before they’re entered into the record.

This ensures that the LLM only *proposes* outputs; automated checks and expert review prevent propagation of errors. Raw, unchecked model suggestions should **never** be finalized without such validation and human oversight.


In [2]:
import pandas as pd

# Define a codebook that maps clinical terms to ICD-10 and SNOMED codes.
CODE_BOOK = {
    "hypertension": {"icd10": "I10", "snomed": "38341003"},
    "type 2 diabetes": {"icd10": "E11.9", "snomed": "44054006"},
    "asthma": {"icd10": "J45.909", "snomed": "195967001"},
    "depression": {"icd10": "F32.9", "snomed": "370143000"},
}

def mock_llm_autocode(note_text):
    """
    Simulates an LLM that proposes medical codes based on input note text.

    Args:
        note_text (str): Clinical note text.

    Returns:
        dict: A dictionary containing code suggestions the LLM might return.
              Each suggestion includes the mention, ICD-10 code, SNOMED code,
              and a model confidence score.
    """
    text = note_text.lower()
    suggestions = []
    # For each term in the code book, check if it appears in the note.
    for term, codes in CODE_BOOK.items():
        if term in text:
            # If found, add a suggestion with a high (simulated) confidence.
            suggestions.append({"mention": term, **codes, "model_confidence": 0.9})
    return {"suggested_codes": suggestions}

# Example patient note for demonstration.
note = (
    "58 y/o with history of hypertension and type 2 diabetes presents for "
    "follow-up. Screen for depression."
)

# Get mock model output for the note.
raw = mock_llm_autocode(note)

print("Raw model output (as an LLM would return):")
import json; print(json.dumps(raw, indent=2))

Raw model output (as an LLM would return):
{
  "suggested_codes": [
    {
      "mention": "hypertension",
      "icd10": "I10",
      "snomed": "38341003",
      "model_confidence": 0.9
    },
    {
      "mention": "type 2 diabetes",
      "icd10": "E11.9",
      "snomed": "44054006",
      "model_confidence": 0.9
    },
    {
      "mention": "depression",
      "icd10": "F32.9",
      "snomed": "370143000",
      "model_confidence": 0.9
    }
  ]
}


### Milestone 1 — validate model output before trusting it

Before using the LLM's output, we must rigorously validate the suggested codes.
This includes checking each suggestion for acceptable ICD-10 codes and ensuring
that the model's confidence is above a minimum threshold. Only suggestions that
meet these criteria should be considered for clinician review; others should be
flagged for further inspection and never automatically entered into the clinical record.

In [3]:
# Define a set of ICD-10 codes that are considered valid for further review.
VALID_ICD = {"I10", "E11.9", "J45.909", "F32.9", "J20.9"}

def validate_autocode(model_output, min_conf=0.5):
    """
    Validates model-suggested codes before they are sent for clinician review.

    Args:
        model_output (dict): Model output containing 'suggested_codes', where each suggestion is a dict
                             with keys including 'icd10' and 'model_confidence'.
        min_conf (float): Minimum model confidence required to accept a suggestion.

    Returns:
        Tuple (pd.DataFrame, pd.DataFrame):
            - DataFrame of accepted suggestions (valid code + confident)
            - DataFrame of flagged suggestions (invalid code or low confidence)
    """
    accepted, flagged = [], []
    for s in model_output["suggested_codes"]:
        # Accept the suggestion if the code is in the allowed set and confidence is high enough.
        ok = (s["icd10"] in VALID_ICD) and (s["model_confidence"] >= min_conf)
        (accepted if ok else flagged).append(s)
    return pd.DataFrame(accepted), pd.DataFrame(flagged)

# Run validation on the mock model's output.
accepted, flagged = validate_autocode(raw)

print("ACCEPTED (valid code + confident) -> queued for clinician review:")
# Show core columns of accepted (validated) suggestions.
print(accepted[["mention", "icd10", "snomed", "model_confidence"]].to_string(index=False))
print(f"\nFlagged/rejected: {len(flagged)}  (would never post automatically)")

ACCEPTED (valid code + confident) -> queued for clinician review:
        mention icd10    snomed  model_confidence
   hypertension   I10  38341003               0.9
type 2 diabetes E11.9  44054006               0.9
     depression F32.9 370143000               0.9

Flagged/rejected: 0  (would never post automatically)


## 16.2 The FHIR + generative AI pattern

When integrating a large language model (LLM) into a clinical environment, the optimal and safest architecture ensures that FHIR (Fast Healthcare Interoperability Resources) continues to serve as the system’s single source of truth for patient data. This process unfolds through several key stages:

1. **Read**: Existing FHIR resources (such as Conditions, Observations, Medications, etc.) are read from the clinical system and form the input data for context assembly.
2. **Context Assembly**: Relevant information is organized and collated—often including specific patient data, observations, and prior notes—into a structured or semi-structured input that is provided to the LLM.
3. **LLM Inference**: The LLM processes the assembled context and generates structured output, typically with suggestions or predictions such as proposed diagnoses, summaries, or potential coding.
4. **Validation and Write-Back**: Critically, before any information generated by the LLM is incorporated into the medical record, its output undergoes validation (e.g., reviewed by a clinician or checked against rules and thresholds). Only after this validation step do approved results get encoded as FHIR resources and written back to the health system.

This process is often visualized as:

```
FHIR resources  ──►  context assembly  ──►  LLM  ──►  structured output
      ▲                                                      │
      └──────────  write back as FHIR (after validation) ────┘
```

Importantly, the LLM should always operate as a supplement to, not a replacement for, the authoritative health record. It should read only from FHIR (ensuring input is up-to-date and governed), and any data it produces must be validated before it is written back as new or updated FHIR resources. This keeps all patient information within the traceable, auditable FHIR workflow—preventing isolated "side channels" of AI output and supporting transparency, accountability, and provenance in clinical data augmentation.


In [4]:
# This function converts each accepted model suggestion into a FHIR Condition resource.
# - FHIR (Fast Healthcare Interoperability Resources) is a healthcare data standard for electronic health records.
# - A "Condition" resource in FHIR encodes a clinical condition (here, an AI-suggested diagnosis or mention).
# - We provide patient context using the patient_id, add SNOMED coding, and tag the resource as "llm-assisted" for auditability.

def to_fhir_condition(row, patient_id="P1001"):
    return {
        "resourceType": "Condition",  # Specify the FHIR resource type.
        "clinicalStatus": {"coding": [{"code": "active"}]},  # Set condition as 'active'.
        "code": {
            "coding": [
                {"system": "http://snomed.info/sct",  # Use SNOMED CT for clinical coding.
                 "code": row["snomed"],              # The predicted clinical code.
                 "display": row["mention"]}          # Human-readable mention from the note.
            ]
        },
        "subject": {"reference": f"Patient/{patient_id}"},  # Link to FHIR Patient resource.
        "meta": {
            "tag": [
                {"code": "llm-assisted", 
                 "display": "AI-suggested, clinician-reviewed"}  # Track AI involvement for audit.
            ]
        }
    }

# Convert all accepted suggestions into FHIR Condition resources.
conditions_fhir = [to_fhir_condition(r) for _, r in accepted.iterrows()]

print("Write-back as FHIR Condition (tagged as AI-assisted for provenance):")

# Show the structure of a generated FHIR Condition resource for reference.
print(json.dumps(conditions_fhir[0], indent=2))

Write-back as FHIR Condition (tagged as AI-assisted for provenance):
{
  "resourceType": "Condition",
  "clinicalStatus": {
    "coding": [
      {
        "code": "active"
      }
    ]
  },
  "code": {
    "coding": [
      {
        "system": "http://snomed.info/sct",
        "code": "38341003",
        "display": "hypertension"
      }
    ]
  },
  "subject": {
    "reference": "Patient/P1001"
  },
  "meta": {
    "tag": [
      {
        "code": "llm-assisted",
        "display": "AI-suggested, clinician-reviewed"
      }
    ]
  }
}


## 16.3 Agentic AI — plan over tools (claims)

A **chatbot** answers in prose. An **agent** is a **controller** that calls
**named tools** with contracts, in a **plan**, and leaves an **audit trail**.
Module 13 built the claims analytics pipeline by hand. Wrapping that pipeline
as an agent is how health systems are starting to expose informatics work to
analysts in plain English — without letting a language model invent an HCC score.

```
  raw claims + eligibility
           │
           ▼
     [clean] → [features: PMPM · HCC] → [risk tiers] → [brief / worklist]
           │                                      ▲
           └──────── ask("who is high risk?") ────┘   (router, not a guessing LLM)
```

**Decision support, not autonomous payment or care.** The agent *recommends*
(risk tiers, outreach lists). A human (care manager, actuary, medical director)
decides. This is not a certified CMS-HCC grouper, a HEDIS engine, or medical advice.

| Pattern | What the model is allowed to do |
|---|---|
| Single LLM call (16.1) | Propose codes from a note — then **validate** |
| FHIR + gen AI (16.2) | Read FHIR, write FHIR **after** review |
| **Agent (this section)** | Choose a **tool**; the tool computes; the model may **rephrase KPIs** |

The engine below is **deterministic and offline**. An optional live model would
only rewrite the briefing — the metrics stay with `kpis()`.


In [ ]:
# Mini claims *book* — the member-year table Module 13 would have produced.
# Kept small so the agent (not the simulator) is the teaching point.
import datetime as dt
import json
import re
import textwrap

rng = np.random.default_rng(7)
n = 80
ages = rng.integers(55, 90, n)
hcc = np.round(0.6 + (ages - 55) * 0.015 + rng.normal(0, 0.15, n), 3).clip(0.4, 2.4)
pmpm = np.round(400 + hcc * 420 + rng.normal(0, 180, n), 2).clip(80, 8000)
er = rng.poisson(0.35, n).clip(0, 5)
ip = rng.poisson(0.20, n).clip(0, 3)
diabetes = rng.binomial(1, 0.28, n)

def _tier(h, e, i):
    if h >= 1.45 or i >= 2 or e >= 3:
        return "4 - Critical"
    if h >= 1.15 or i >= 1 or e >= 2:
        return "3 - High"
    if h >= 0.9:
        return "2 - Rising"
    return "1 - Low"

book = pd.DataFrame({
    "member_id": [f"M{2000+i}" for i in range(n)],
    "age": ages,
    "hcc": hcc,
    "pmpm": pmpm,
    "er_visits": er,
    "ip_visits": ip,
    "has_diabetes": diabetes,
    "glucose_tested": rng.binomial(1, 0.72, n) * diabetes,  # only diabetics can "count"
})
book["risk_tier"] = [_tier(h, e, i) for h, e, i in zip(book["hcc"], book["er_visits"], book["ip_visits"])]
# Diabetics without a glucose test are the care-gap list (HEDIS-like, educational)
book["care_gap"] = (book["has_diabetes"] == 1) & (book["glucose_tested"] == 0)

print(f"Agent book: {len(book)} members · mean PMPM ${book['pmpm'].mean():,.0f} · mean HCC {book['hcc'].mean():.3f}")
print(book["risk_tier"].value_counts().sort_index().to_string())


In [ ]:
# ClaimsInformaticsAgent: a thin controller.
#   run()  = compute KPIs from the book (the "plan")
#   ask()  = keyword router onto tools  (not an LLM inventing numbers)
#   _log() = audit trail so you can reconstruct what ran

class ClaimsInformaticsAgent:
    def __init__(self, book: pd.DataFrame):
        self.book = book.copy()
        self.focus = None          # last member_id named (conversation memory)
        self.audit = []
        self._kpis = None

    def _log(self, tool, detail=""):
        self.audit.append({
            "ts": dt.datetime.now().isoformat(timespec="seconds"),
            "tool": tool,
            "detail": str(detail)[:180],
        })

    def kpis(self) -> dict:
        df = self.book
        high = df["risk_tier"].isin(["3 - High", "4 - Critical"])
        top5 = df.nlargest(max(1, int(len(df) * 0.05)), "pmpm")
        return {
            "n_members": int(len(df)),
            "avg_age": float(df["age"].mean()),
            "avg_pmpm": float(df["pmpm"].mean()),
            "avg_hcc": float(df["hcc"].mean()),
            "high_critical": int(high.sum()),
            "high_critical_pct": float(high.mean() * 100),
            "top5_cost_share": float(top5["pmpm"].sum() / df["pmpm"].sum() * 100),
            "care_gap_n": int(df["care_gap"].sum()),
            "n_diabetes": int(df["has_diabetes"].sum()),
        }

    def run(self):
        """Fixed plan: feature table is already here; freeze KPIs for the briefing."""
        self._log("run", "freeze KPIs from member-year book")
        self._kpis = self.kpis()
        return self

    def briefing(self) -> str:
        k = self._kpis or self.kpis()
        return textwrap.dedent(f"""
        CLAIMS INFORMATICS — LEADERSHIP BRIEFING (synthetic)
        ----------------------------------------------------
        Population   {k['n_members']} members · mean age {k['avg_age']:.1f}
        Spend        ${k['avg_pmpm']:,.2f} mean PMPM
        Risk         mean HCC {k['avg_hcc']:.3f} (1.0 would be a real book's average)
                     High/Critical {k['high_critical']} ({k['high_critical_pct']:.1f}%)
        Concentration top 5% of members = {k['top5_cost_share']:.1f}% of PMPM mass
        Quality      {k['care_gap_n']} of {k['n_diabetes']} diabetics missing a glucose test

        Recommended actions (human review required)
          1. Assign High/Critical members to intensive case management.
          2. Work the glucose-test gap list, highest PMPM first.
          3. Do not treat these figures as a CMS settlement or HEDIS submission.
        """).strip()

    def worklist(self, n=8) -> pd.DataFrame:
        df = self.book[self.book["risk_tier"].str.startswith(("3", "4"))]
        cols = ["member_id", "age", "risk_tier", "pmpm", "hcc", "er_visits", "ip_visits"]
        return df.nlargest(n, "pmpm")[cols]

    def ask(self, question: str) -> str:
        q = question.strip()
        ql = q.lower()
        self._log("ask", q)

        member_key = None
        m = re.search(r"\bM\d{4}\b", q, re.I)
        if m:
            member_key = m.group(0).upper()
            self.focus = member_key
        elif self.focus and re.search(r"\b(it|its|this member|that member)\b", ql):
            member_key = self.focus

        if member_key:
            row = self.book.loc[self.book["member_id"] == member_key]
            if row.empty:
                return f"{member_key} is not in this book."
            r = row.iloc[0]
            return (f"{r.member_id} · age {int(r.age)} · {r.risk_tier}\n"
                    f"HCC {r.hcc:.2f} · PMPM ${r.pmpm:,.2f} · ER {int(r.er_visits)} · IP {int(r.ip_visits)}\n"
                    f"(Recommendation only — a clinician confirms any outreach.)")

        if re.search(r"care management|high risk|critical|worklist|who needs|outreach", ql):
            wl = self.worklist()
            lines = ["Care-management worklist (High/Critical, highest PMPM first)",
                     "Human review required before outreach.\n"]
            for _, r in wl.iterrows():
                lines.append(f"  {r.member_id}  {r.risk_tier}  PMPM ${r.pmpm:,.0f}  HCC {r.hcc:.2f}")
            return "\n".join(lines)

        if re.search(r"care gap|glucose|quality|diabetes", ql):
            g = self.book[self.book["care_gap"]].nlargest(6, "pmpm")
            lines = [f"Glucose-test gap: {int(self.book['care_gap'].sum())} diabetic members\n"]
            for _, r in g.iterrows():
                lines.append(f"  {r.member_id}  {r.risk_tier}  ${r.pmpm:,.0f} PMPM")
            return "\n".join(lines)

        if re.search(r"brief|summary|kpi|leadership", ql):
            return self.briefing()

        if re.search(r"\bhelp\b", ql):
            return ("Ask: 'who needs care management?', 'care gaps', "
                    "'brief leadership', or 'profile M2000'.")

        return "I could not route that. Try 'help'."

agent = ClaimsInformaticsAgent(book).run()
print("Pipeline complete. KPIs frozen:")
print(json.dumps(agent._kpis, indent=2))
print("Audit:", [a["tool"] for a in agent.audit])


### Milestone 2 — ask the agent in plain English

`ask()` is a **router**: keywords → tool → formatted string. "What's its HCC?"
uses **conversation memory** (the last member you named). A production system
could swap in Claude to *phrase* the same payload — not to compute it.


In [ ]:
# Scripted transcript — the questions a plan analyst actually types
top = agent.book.nlargest(1, "pmpm").iloc[0]["member_id"]
questions = [
    "Who needs care management?",
    "Where are our quality / care gaps?",
    f"Profile {top}",
    "What's its HCC?",
    "Brief leadership.",
]
for q in questions:
    print("=" * 68)
    print("Q:", q)
    print("-" * 68)
    print(agent.ask(q))
    print()


**Grounded narrative.** If a language model writes the memo, it receives the
KPI **JSON**. It must not invent a PMPM, member, or savings figure. Offline we
use a deterministic "rewrite" that only templates those JSON fields — the same
contract as a live call.


In [ ]:
def mock_llm_memo(kpis: dict) -> str:
    """Stand-in for a live LLM. May only interpolate values from kpis()."""
    return textwrap.dedent(f"""
    MEMO (model-phrased; numbers from tools only)

    We reviewed a synthetic book of {kpis['n_members']} members (mean age
    {kpis['avg_age']:.1f}). Mean PMPM is ${kpis['avg_pmpm']:,.2f}; mean HCC is
    {kpis['avg_hcc']:.3f}. High/Critical accounts for {kpis['high_critical']}
    members ({kpis['high_critical_pct']:.1f}%). The top 5% concentrate
    {kpis['top5_cost_share']:.1f}% of PMPM mass. {kpis['care_gap_n']} of
    {kpis['n_diabetes']} members with diabetes have no glucose test on file.

    These are recommendations for human review, not payment or care orders.
    """).strip()

print(mock_llm_memo(agent.kpis()))
print()
print("Audit trail (who asked, which tool, when):")
print(pd.DataFrame(agent.audit).to_string(index=False))


**Guardrails worth copying** (from the claims-agent pattern in Module 13's companion notebook):

| Guardrail | How this agent does it |
|---|---|
| **Synthetic / no PHI** | Generated members only |
| **Human in the loop** | Worklists are recommendations |
| **Transparent rules** | Risk tiers are explicit `if` statements |
| **Audit trail** | `agent.audit` records `run` / `ask` |
| **Grounded narrative** | Memo interpolates `kpis()` JSON; it does not recompute HCC |
| **Not a certified engine** | Not CMS-HCC, not HEDIS, not medical advice |


## 16.4 Federated analytics – move the computation, not the data

In healthcare, data privacy laws and regulatory requirements frequently prevent the movement or pooling of actual patient records across different institutions or clinical sites. As a result, organizations often cannot simply combine all their raw data into a single, centralized database for analysis.

**Federated analytics** is a privacy-preserving solution to this challenge. In a federated approach, each participating site (for example, hospitals or clinics) keeps its patient records locally. Instead of sharing sensitive, row-level data, each site computes **local summary statistics** (such as totals, means, or other aggregates) on its own data, and then only these statistics are communicated to a central coordinator or aggregated further. No individual patient-level data ever leaves the institution where it originated, greatly reducing privacy risks.

For example, to calculate the average Hemoglobin A1c (HbA1c, a marker of blood sugar control) across three simulated sites, each site would:
- Compute the sum and count of their patients’ A1c values locally,
- Share only (sum, count) pairs, not any raw measurements or records,
- The central coordinator then combines these numbers to compute the overall mean, without ever seeing any patient’s full data.

This paradigm supports privacy, regulatory compliance, and cross-institutional collaboration. The code below demonstrates a federated mean A1c computation using simulated site splits.


In [5]:
import numpy as np

# Extract the Hemoglobin A1c (HbA1c) observation values from the EHR data.
obs = ehr["observations"]
a1c = obs[obs["observation"] == "Hemoglobin A1c"]["value"].to_numpy()

# Create a random number generator with a fixed seed for reproducibility.
rng = np.random.default_rng(1)

# Randomly permute and split the A1c values into 3 "sites". 
# Each site gets a portion of the data and never shares raw patient-level rows with others.
sites = np.array_split(rng.permutation(a1c), 3)

# For federated analytics, each site locally computes aggregate statistics:
# Only the 'sum' of values and the 'count' of patients are shared externally; no raw values leave the site.
local_stats = [
    {
        "site": i + 1,      # Identifier for the site (1, 2, or 3)
        "n": len(s),        # Number of records at this site
        "sum": float(s.sum())  # Sum of A1c values at this site (converted to float)
    }
    for i, s in enumerate(sites)
]

print("Per-site shared statistics (no raw values leave the site):")
for st in local_stats:
    print(" ", st)

# The central coordinator receives only the aggregates, not any individual records,
# and computes the overall (global) mean A1c by combining per-site sums and counts.
global_n = sum(st["n"] for st in local_stats)
global_mean = sum(st["sum"] for st in local_stats) / global_n

print(f"\nFederated global mean A1c: {global_mean:.3f} % (n={global_n})")

# For validation, calculate the mean as if centralizing all data (i.e., if pooling were allowed).
# This should match the federated result if all math is correct.
print(f"Centralized check (if pooling were allowed): {a1c.mean():.3f} %  -> identical")

Per-site shared statistics (no raw values leave the site):
  {'site': 1, 'n': 119, 'sum': 845.7000000000002}
  {'site': 2, 'n': 119, 'sum': 830.6000000000001}
  {'site': 3, 'n': 118, 'sum': 851.9000000000001}

Federated global mean A1c: 7.102 % (n=356)
Centralized check (if pooling were allowed): 7.102 %  -> identical


### Milestone 3 - an equity / bias check

This step involves performing an equity or bias check to assess whether our screening flag is applied fairly across different demographic groups (e.g., by sex). We compare the flagging rate in each group to look for large disparities, which could indicate bias in the data or model. If a significant gap is found, it may require further investigation and mitigation to ensure fair and equitable model performance.

In [6]:
# The following code assesses whether the A1c screening flag (which indicates possible diabetes) is being applied equitably across different sex groups.

# 1. Retrieve patient table and A1c observation data.
patients = ehr["patients"]
# 2. For each patient, find their highest Hemoglobin A1c (HbA1c) value.
a1c_max = obs[obs["observation"] == "Hemoglobin A1c"].groupby("patient_id")["value"].max()
# 3. Merge this maximum A1c value with the patient demographic table using patient_id as the index.
df = patients.set_index("patient_id").assign(a1c=a1c_max)
# 4. Create a 'flagged' column: patients are flagged if their A1c is 6.5 or above (possible diabetes threshold).
df["flagged"] = (df["a1c"] >= 6.5)

# 5. For each sex group, calculate the screening flag rate (percentage of patients flagged) and the subgroup count.
equity = (
    df.dropna(subset=["a1c"])                     # Exclude patients with missing A1c
      .groupby("sex")["flagged"]                  # Group by sex, look at "flagged" column
      .agg(["mean", "count"])                     # Calculate mean (flag rate) and count
      .rename(columns={"mean": "flag_rate"})      # Rename 'mean' to 'flag_rate'
)
# 6. Convert flag rates to percentages (rounded to 1 decimal).
equity["flag_rate"] = (equity["flag_rate"] * 100).round(1)

print("Screening-flag rate by sex (watch for large gaps):")
print(equity.to_string())

# 7. Calculate the absolute difference in flag rates between the two sex groups.
gap = abs(equity["flag_rate"].iloc[0] - equity["flag_rate"].iloc[1])

# 8. Print a message interpreting the gap.
print(
    f"\nAbsolute gap: {gap:.1f} percentage points "
    f"-> {'review for bias' if gap > 10 else 'within a reasonable band (synthetic data)'}"
)

Screening-flag rate by sex (watch for large gaps):
        flag_rate  count
sex                     
female       80.6     98
male         81.7     82

Absolute gap: 1.1 percentage points -> within a reasonable band (synthetic data)


## 16.5 Ethics, regulation, and equity

The technical work is necessary but not sufficient. Responsible health AI/informatics
weighs:

- **Regulation** - FDA oversight of AI/ML as software-as-a-medical-device (SaMD);
  HIPAA; emerging AI-specific rules.
- **Transparency** - model cards, documented training data, explainability where
  decisions affect care.
- **Equity** -  auditing for disparate performance across race, sex, age, language,
  and socioeconomic status; representative training data.
- **Human oversight** - AI (including **agents** that call tools) augments
  clinicians and analysts; it does not replace accountable human judgment.
  Worklists and risk scores are recommendations, not orders.

These are entry points, not the last word — each is an active area your work will
keep engaging with.


## 16.6 Data Governance

Data governance is the framework of processes, policies, and standards that ensure health data is accurate, secure, and used appropriately throughout its lifecycle.

Key elements of data governance in health informatics include:

- **Data Quality**: Ensuring that data is complete, accurate, timely, and consistent.
- **Privacy and Security**: Protecting patient information to comply with laws like HIPAA; applying access controls, audits, and de-identification where needed.
- **Data Stewardship and Ownership**: Defining responsibilities for data management, from custodianship to authorized access and consent management.
- **Lineage and Provenance**: Tracking where data comes from, how it is transformed, and recording any changes for full auditability.
- **Compliance**: Meeting regulatory obligations and best practices for data handling, retention, sharing, and disposal.
- **Interoperability Standards**: Using shared vocabularies (like FHIR, SNOMED, LOINC) and data structures for safe and seamless data exchange between systems.

Good data governance builds trust, ensures ethical use, and is essential for responsible health AI development and collaboration across institutions.

## Exercises

1. Extend the mock autocoder to return a **low-confidence** suggestion and confirm
   the validator routes it to *flagged*, not *accepted*.
2. Add a third site with a deliberately **different** A1c distribution and confirm
   the federated mean still matches the pooled mean exactly.
3. Extend the equity check to **age bands** as well as sex, and report the largest
   subgroup gap.
4. Add an `ask()` route for **"what is the mean HCC?"** that reads `kpis()` —
   do not hard-code a number in the reply string.



## Key takeaways

- LLMs assist documentation and **autocoding**, but the pattern is always
  **extract → structure → validate → human review**.
- Keep **FHIR as source of truth**: LLMs read from and write back to it, tagged for provenance.
- **Agentic AI** is a **plan over tools** (clean → features → risk → brief), not a
  chatbot inventing PMPM. Numbers stay with code; the model may only rephrase.
- **Federated** analytics shares **statistics, not records** — and can match pooled results exactly.
- **Ethics, regulation, and equity** decide what should ship, not just what can.

---



## 🎓 Series complete

That closes **Health Informatics in Python** — 16 modules across 4 parts, from the
information lifecycle through standards, interoperability, data engineering, privacy,
NLP, decision support, population health, claims analysis, warehousing, bulk exchange,
and emerging AI (including agentic workflows). Every module runs on self-contained
synthetic data, so the whole series is reproducible end to end with no PHI and no
external downloads.
